<a href="https://colab.research.google.com/github/jccrews256/ST-554-HW-7/blob/main/Homework%207%20Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ST 554 Homework 4
*By: Cass Crews*

In this notebook, we will explore the process of developing machine learning models to predict future values of some response variable of interest. In particular, we will fit a final predictive model for both a regression problem and a classification problem while considering multiple model classes for each.

While the goal of the notebook is the process of developing a predictive model rather than to actually develop a model for some use case, we still need an example dataset. We will use the [Wine Quality dataset](https://archive.ics.uci.edu/dataset/186/wine+quality) from the UC-Irvine Machine Learning Repository. This dataset documents several characteristics of nearly 7,000 wines from northern Portugal.

For the regression problem, our goal is to construct a model that adequately predicts the alcohol level of a wine given its other characteristics. We will consider the following model classes:
- Multiple linear regression models
- LASSO regression models
- Ridge regression models
- Elastic Net regression models.

To obtain the final model, will first split our dataset of wines into a training and test set. We will then use 10-fold cross-validation on the training set to identify the best model from each class. The final step will be to use the top-performing model from each class to predict alcohol content for the test set, allowing us to identify the best-performing overall model.

For the classification problem, our goal is to construct a model that adequately predicts whether a wine is red or white given its other characteristics. In this case, we will consider:
- Logistic regression models
- Logistic regression models with a LASSO penalty
- Logistic regression models with a ridge penalty
- Logistic regression models with an elastic net penalty.

Note that while logistic regression includes the term "regression," each of the four classes above are classification models from the machine learning perspective. That is, the response is categorical rather than a continuous numeric variable. The steps to obtain a final classification model will be the same steps used for the regression problem.

## Preparatory Steps

Before we begin tuning models, we need to read in the necessary modules and data. We'll start with the modules:

In [2]:
import pandas as pd
import numpy as np
from sklearn import linear_model
from math import sqrt
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, LassoCV, Lasso, RidgeCV, \
Ridge, ElasticNetCV, ElasticNet

We're now ready to read in the data. The white and red wines are stores in separately csv files, so we will combine both wine types into a single `pandas` dataframe. In doing so, we will create a dummy variable that indicates whether each wine is `red`, with a value of 1 if so and 0 if not.

*Note to the reader: You will need to store the two data files in your local environment to run the remaining code chunks in this notebook.*

In [7]:
# Reading in the red wine data and adding red indicator
red_wines = pd.read_csv("winequality-red.csv", sep = ";")
red_wines["red"] = 1

# Reading in the white wine data and adding red indicator
white_wines = pd.read_csv("winequality-white.csv", sep = ";")
white_wines["red"] = 0

# Concatenating the two dataframes
wines = pd.concat([red_wines, white_wines], ignore_index = True)

Let's figure out how many red and white wines are in the dataset.

In [22]:
wines.red.value_counts()

,count
red,
0,4898
1,1599


There are roughly three times as many white wines as red wines. This is something we'll need to keep in mind when we split the data into training and test sets, as this class imbalance could lead to us accidentally creating a training set with predominantly white wines.

Let's print the first few rows to see the candidate predictors available to us for our modeling problems.

In [8]:
# Extracting first five rows
wines.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,red
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,1
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,1
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1


we have a variety of characteristics for each wine, ranging from `fixed acidity` to `density` and sulphate levels.

An immediate concern for this dataframe is that the variable names include spaces. This can make it difficult to extract individual variables, so let's replace each space with an underscore.

In [17]:
# Replacing spaces with underscores in column names
wines.columns = [col.replace(" ","_") for col in wines.columns.to_list()]

# Extracting first few rows
wines.head()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality,red
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,1
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,1
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1


These names are much more manageable.

The last step before we move to the regression problem is to confirm there are no missing values for any of the columns.

In [18]:
# Checking for missing values
wines.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed_acidity         6497 non-null   float64
 1   volatile_acidity      6497 non-null   float64
 2   citric_acid           6497 non-null   float64
 3   residual_sugar        6497 non-null   float64
 4   chlorides             6497 non-null   float64
 5   free_sulfur_dioxide   6497 non-null   float64
 6   total_sulfur_dioxide  6497 non-null   float64
 7   density               6497 non-null   float64
 8   pH                    6497 non-null   float64
 9   sulphates             6497 non-null   float64
 10  alcohol               6497 non-null   float64
 11  quality               6497 non-null   int64  
 12  red                   6497 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 660.0 KB


It seems there are no missing values. The dataset web page (linked above) also indicates there are no missing values, but it's always best to check!

## Regression Problem

We are now ready to develop a prediction model for `alcohol` level. As we discussed in the introduction, we will consider candidate models across four different model classes:
- Multiple linear regression models
- LASSO regression models
- Ridge regression models
- Elastic Net regression models.

#### Preparing Test and Training Sets

The first step is to separate the full `wines` dataset into training and test sets. We will reserve 20\% for the test set and the rest for the training set. As there are many more white wines than red wines and we don't want to accidentally train our models using predominantly white wines, we will use stratified sampling to ensure the proportions for each wine type are roughly equal across training and test sets.

In [23]:
# Splitting into training and test sets
# Stratifying based on red (wine type)
X_train, X_test, y_train, y_test = train_test_split(
    wines.drop("alcohol", axis = 1),
    wines.alcohol,
    test_size = 0.20,
    stratify = wines.red,
    random_state = 5
)

As three of the classes we will consider involve coefficient shrinkage, we need to standardize each predictor to ensure differences in scale across predictors are not influencing the relative shrinkage of the predictors' coefficient estimates. We will save the predictor means and standard deviations from the training set so that we can apply the same standardizing transformations to the test set. This ensures the training and test sets are on the same scale.

In [25]:
# Capturing means and standard deviations for predictors
means = X_train.apply(np.mean, axis = 0)
stds = X_train.apply(np.std, axis = 0, ddof = 1) # n-1 degrees of freedom

Now that we've saved the training set means and standard deviations, let's standardized the training set predictors.

In [33]:
# Standardizing the training set predictors
X_train = X_train.apply(lambda x: (x - np.mean(x, axis = 0))/
                        np.std(x, axis = 0, ddof = 1))

Let's now transform the test set predictors using the training set means and standard deviations.

In [34]:
# Creating basic function to standardize using supplied mean and standard dev.
def std_col(col, mean, std):
    return (col-mean)/std

# Loop through predictors and standardize for test set
for c in X_test.columns:
    X_test[c] = std_col(X_test[c], means[c], stds[c])

To highlight the fact that we are standardizing candidate predictors in the training and test sets using training set means and standard deviations, we can extract the variables' rescaled means for each set.

In [41]:
pd.DataFrame([X_train.mean(), X_test.mean()], index = ["train_mean", "test_mean"])

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,quality,red
train_mean,1.093774e-17,3.007878e-17,3.281321e-17,-2.871156e-17,0.000000,-1.093774e-17,-3.554765e-17,2.187547e-17,-1.230495e-17,0.000000,-4.648538e-17,-1.093774e-17
test_mean,-3.063494e-02,-3.260385e-02,1.552617e-03,-3.463458e-02,-0.019631,3.664099e-02,-2.107743e-02,-4.478672e-02,3.245976e-02,-0.006491,3.659811e-02,1.168337e-04


Note that the training set means are all effectively zero, while the test set means are close to zero but not effectively zero.

#### Tuning Multiple Linear Regression

We are now ready to tune the multiple linear regression (MLR) model class to identify the "best" model for predicting `alcohol`. To do so, let's consider four different parametrizations:
- $E(alcohol|X) = \beta_0 + \beta_1sulphates + \beta_2residual\_sugar$
- $E(alcohol|X) = \beta_0 + \beta_1sulphates + \beta_2residual\_sugar + \beta_3pH$
- $E(alcohol|X) = \beta_0 + \beta_1sulphates + \beta_2residual\_sugar + \beta_3pH + \beta_4sulphates\cdot residual\_sugar + \beta_5sulphates\cdot pH + \beta_6residual\_sugar\cdot pH$
- $E(alcohol|X) = \beta_0 + \beta_1sulphates + \beta_2residual\_sugar + \beta_3pH + \beta_4sulphates\cdot residual\_sugar + \beta_5sulphates\cdot pH + \beta_6residual\_sugar\cdot pH + \beta_7sulphates^2 + \beta_8residual\_sugar^2 + \beta_9pH^2$

Why have I chosen these models? Because I actually know what `sulphates`, `residual_sugar`, and `pH` mean in the context of wine, something I cannot say for the other candidate predictors....
